# Hodgkin's Razor

**Which molecular mechanism did the compound move?**

Two recordings of one well go in. What comes out is the mechanism, with an
effect size, a credible interval and a calibrated probability.

This notebook runs on CPU. It uses the trained model in `models/twin` and the
recording pairs bundled in `data/examples`, which are extracts of the Tampere
comparative MEA dataset (CC BY 4.0). Nothing here needs a GPU; rebuilding the
simulation bank does.

The model is trained only on simulations and has never seen a compound label.


In [ ]:
import sys, json, pathlib
import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'hodgkins_razor').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hodgkins_razor import features as F, nde, params as P, report
print('features:', F.N_FEATURE, ' parameters:', P.N_PARAM,
      ' mechanisms a compound can move:', P.N_SHIFT)


## 1. Load a recording pair

Each bundled file holds the spike times of one well before and after wash-on,
in the same 60 second window.


In [ ]:
name = 'rat_cnqx'   # try human_gabazine, rat_gaba, rat_ttx, rat_control
d = json.loads((ROOT / 'data' / 'examples' / f'{name}.json').read_text())
base = np.array(d['baseline']).reshape(-1, 2)
treat = np.array(d['treated']).reshape(-1, 2)
dur = float(d['duration'])
print(d['label'])
print(f"{len(base)} events before, {len(treat)} after, over {dur:.0f} s")


## 2. Summary statistics

The same function is used for a simulation and for a recording, so a
difference between them can never come from two implementations.


In [ ]:
x_base = F.compute(base, 16, dur)
x_treat = F.compute(treat, 16, dur)
for n, a, b in list(zip(F.NAMES, x_base, x_treat))[:8]:
    print(f'{n:22s} {a:9.3f} -> {b:9.3f}')


## 3. The posterior

A conditional normalising flow gives the joint posterior over the culture
parameters and the shift the compound caused. A separate calibrated head gives,
per mechanism, the probability that it moved at all.


In [ ]:
twin = nde.Twin.load(ROOT / 'models' / 'twin', device='cpu')
post = twin.posterior(x_base, x_treat, n_samples=4000)
rows = nde.summarise(post)
for r in rows:
    size = f"{r['effect']:.2f}x" if r['kind'] == 'fold' else f"{r['effect']:+.2f}"
    print(f"{r['label']:28s} p={r['p_active']:.2f}  {size:>9s}  {r['target']}")


## 4. The report

The predictive check needs a CUDA device, so it is skipped here and the report
says so rather than implying the recording passed it.


In [ ]:
body = report.build(post, None, x_base, x_treat, meta={'duration_s': dur})
print(body['sentence'])
print()
for c in body['classes']:
    print(f"  {c['name']:38s} {c['probability']:.2f}")


## 5. Every bundled pair

The answer key comes from the published pharmacology of each compound and is
used only for scoring. The model never sees it.


In [ ]:
KEY = {'CNQX': 'g_ampa', 'D-AP5': 'g_nmda', 'GABA': 'g_gaba',
       'gabazine': 'g_gaba', 'kainic acid': 'g_ampa', 'TTX': 'g_na',
       'vehicle control': None}
hits = total = 0
for p in sorted((ROOT / 'data' / 'examples').glob('*.json')):
    e = json.loads(p.read_text())
    xb = F.compute(np.array(e['baseline']).reshape(-1, 2), 16, e['duration'])
    xt = F.compute(np.array(e['treated']).reshape(-1, 2), 16, e['duration'])
    r = nde.summarise(twin.posterior(xb, xt, n_samples=2000))[0]
    want = KEY.get(e['compound'])
    called = r['key'] if r['p_active'] >= 0.5 else '-'
    if want:
        total += 1
        hits += called == want
    mark = '+' if (called == want if want else called == '-') else '.'
    print(f"{e['label']:44s} {called:9s} p={r['p_active']:.2f}  expected {str(want):9s} {mark}")
print()
print(f'top-1 on these examples: {hits}/{total}   chance {1/P.N_SHIFT:.2f}')


## 6. What is not in this notebook

* The predictive check and the guard, which re-simulate the twin and need CUDA.
* The full pre-registered evaluation over every well, which is
  `python scripts/evaluate.py` and writes `results/RESULTS.md`.
* The two-compartment chip study, `python scripts/chip_study.py`.

The measured limits are in `results/RESULTS.md` and the technical report.
Among nine mechanisms this problem is hard, and the probability attached to
every call is what carries that.
